In [3]:
# ============================================================
# MONTHLY TEMPERATURE vs MTBF ANALYSIS
# ============================================================
# DATASET:
# PCB_MTBF_Processed.xlsx
#
# ANALYSIS INCLUDED:
#
# 1. Region-wise Monthly Correlation
# 2. SLoc-wise Monthly Correlation
# 3. Pearson Correlation
# 4. Spearman Correlation
# 5. Kendall Tau Correlation
# 6. ANOVA
# 7. Kruskal Wallis
# 8. Tukey HSD
# 9. Chi Square Test
# 10. Cramer's V
#
# ============================================================

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

from scipy.stats import (
    pearsonr,
    spearmanr,
    kendalltau,
    f_oneway,
    kruskal,
    chi2_contingency
)

from statsmodels.stats.multicomp import pairwise_tukeyhsd

warnings.filterwarnings("ignore")

# ============================================================
# CREATE OUTPUT FOLDERS
# ============================================================

base_output_folder = r"MTBF/Temperature_Analysis/Monthly_Analysis"

folders = [
    "Region_Wise",
    "SLoc_Wise",
    "Correlation_Results",
    "ANOVA",
    "Kruskal_Wallis",
    "Tukey_HSD",
    "Chi_Square",
    "Cramers_V"
]

for folder in folders:
    os.makedirs(
        os.path.join(base_output_folder, folder),
        exist_ok=True
    )

print("Folders Created Successfully")

# ============================================================
# LOAD DATASET
# ============================================================

file_path = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\100\MTBF\PCB_MTBF_Processed.xlsx"

df = pd.read_excel(file_path)

print("\nDataset Loaded Successfully")
print("Shape :", df.shape)

# ============================================================
# COLUMN STANDARDIZATION
# ============================================================

df.columns = df.columns.str.strip()

print("\nColumns:")
print(df.columns.tolist())

# ============================================================
# REQUIRED COLUMNS
# ============================================================

required_columns = [
    'Region',
    'SLoc',
    'Month',
    'Year',
    'MTBF_Days',
    'Tavg',
    'Temperature_Band',
    'MTBF_Category'
]

# ============================================================
# CHECK REQUIRED COLUMNS
# ============================================================

missing_cols = [
    col for col in required_columns
    if col not in df.columns
]

if len(missing_cols) > 0:
    print("\nMissing Columns:")
    print(missing_cols)

else:
    print("\nAll Required Columns Present")

# ============================================================
# REMOVE MISSING VALUES
# ============================================================

df = df.dropna(
    subset=[
        'MTBF_Days',
        'Tavg',
        'Region',
        'SLoc'
    ]
)

print("\nAfter Removing Missing Values:")
print(df.shape)

# ============================================================
# SAVE CLEANED DATASET
# ============================================================

cleaned_output = os.path.join(
    base_output_folder,
    "Temperature_Monthly_Cleaned_Data.xlsx"
)

df.to_excel(cleaned_output, index=False)

print("\nCleaned Dataset Saved")

# ============================================================
# MONTHLY REGION-WISE ANALYSIS DATASET
# ============================================================

region_monthly = (
    df.groupby(
        ['Region', 'Year', 'Month']
    )
    .agg({
        'MTBF_Days': 'mean',
        'Tavg': 'mean'
    })
    .reset_index()
)

region_monthly_output = os.path.join(
    base_output_folder,
    "Region_Wise",
    "Region_Wise_Monthly_Data.xlsx"
)

region_monthly.to_excel(
    region_monthly_output,
    index=False
)

print("\nRegion-Wise Monthly Dataset Saved")

# ============================================================
# MONTHLY SLOC-WISE ANALYSIS DATASET
# ============================================================

sloc_monthly = (
    df.groupby(
        ['SLoc', 'Year', 'Month']
    )
    .agg({
        'MTBF_Days': 'mean',
        'Tavg': 'mean'
    })
    .reset_index()
)

sloc_monthly_output = os.path.join(
    base_output_folder,
    "SLoc_Wise",
    "SLoc_Wise_Monthly_Data.xlsx"
)

sloc_monthly.to_excel(
    sloc_monthly_output,
    index=False
)

print("\nSLoc-Wise Monthly Dataset Saved")

# ============================================================
# CORRELATION FUNCTION
# ============================================================

def correlation_analysis(
    dataframe,
    group_column,
    output_filename
):

    results = []

    grouped = dataframe.groupby(group_column)

    for group_name, group_data in grouped:

        if len(group_data) < 3:
            continue

        x = group_data['Tavg']
        y = group_data['MTBF_Days']

        # ----------------------------------------------------
        # PEARSON
        # ----------------------------------------------------

        try:
            pearson_corr, pearson_p = pearsonr(x, y)

        except:
            pearson_corr, pearson_p = np.nan, np.nan

        # ----------------------------------------------------
        # SPEARMAN
        # ----------------------------------------------------

        try:
            spearman_corr, spearman_p = spearmanr(x, y)

        except:
            spearman_corr, spearman_p = np.nan, np.nan

        # ----------------------------------------------------
        # KENDALL TAU
        # ----------------------------------------------------

        try:
            kendall_corr, kendall_p = kendalltau(x, y)

        except:
            kendall_corr, kendall_p = np.nan, np.nan

        # ----------------------------------------------------
        # STORE RESULTS
        # ----------------------------------------------------

        results.append({

            group_column: group_name,

            'Pearson_Correlation': pearson_corr,
            'Pearson_P_Value': pearson_p,

            'Spearman_Correlation': spearman_corr,
            'Spearman_P_Value': spearman_p,

            'KendallTau_Correlation': kendall_corr,
            'KendallTau_P_Value': kendall_p
        })

    results_df = pd.DataFrame(results)

    save_path = os.path.join(
        base_output_folder,
        "Correlation_Results",
        output_filename
    )

    results_df.to_excel(save_path, index=False)

    print(f"\nCorrelation Results Saved : {output_filename}")

# ============================================================
# REGION-WISE CORRELATION
# ============================================================

correlation_analysis(
    region_monthly,
    'Region',
    'Region_Wise_Correlation.xlsx'
)

# ============================================================
# SLOC-WISE CORRELATION
# ============================================================

correlation_analysis(
    sloc_monthly,
    'SLoc',
    'SLoc_Wise_Correlation.xlsx'
)

# ============================================================
# ANOVA FUNCTION
# ============================================================

def perform_anova(
    dataframe,
    group_column,
    output_filename
):

    grouped_data = []

    for name, group in dataframe.groupby(group_column):

        values = group['MTBF_Days'].dropna()

        if len(values) > 1:
            grouped_data.append(values)

    if len(grouped_data) < 2:
        print(f"Not enough groups for ANOVA : {group_column}")
        return

    f_stat, p_value = f_oneway(*grouped_data)

    anova_df = pd.DataFrame({

        'F_Statistic': [f_stat],
        'P_Value': [p_value]

    })

    save_path = os.path.join(
        base_output_folder,
        "ANOVA",
        output_filename
    )

    anova_df.to_excel(save_path, index=False)

    print(f"\nANOVA Saved : {output_filename}")

# ============================================================
# REGION-WISE ANOVA
# ============================================================

perform_anova(
    region_monthly,
    'Region',
    'Region_Wise_ANOVA.xlsx'
)

# ============================================================
# SLOC-WISE ANOVA
# ============================================================

perform_anova(
    sloc_monthly,
    'SLoc',
    'SLoc_Wise_ANOVA.xlsx'
)

# ============================================================
# KRUSKAL WALLIS FUNCTION
# ============================================================

def perform_kruskal(
    dataframe,
    group_column,
    output_filename
):

    grouped_data = []

    for name, group in dataframe.groupby(group_column):

        values = group['MTBF_Days'].dropna()

        if len(values) > 1:
            grouped_data.append(values)

    if len(grouped_data) < 2:
        print(f"Not enough groups for Kruskal : {group_column}")
        return

    h_stat, p_value = kruskal(*grouped_data)

    kruskal_df = pd.DataFrame({

        'H_Statistic': [h_stat],
        'P_Value': [p_value]

    })

    save_path = os.path.join(
        base_output_folder,
        "Kruskal_Wallis",
        output_filename
    )

    kruskal_df.to_excel(save_path, index=False)

    print(f"\nKruskal Saved : {output_filename}")

# ============================================================
# REGION-WISE KRUSKAL
# ============================================================

perform_kruskal(
    region_monthly,
    'Region',
    'Region_Wise_Kruskal.xlsx'
)

# ============================================================
# SLOC-WISE KRUSKAL
# ============================================================

perform_kruskal(
    sloc_monthly,
    'SLoc',
    'SLoc_Wise_Kruskal.xlsx'
)

# ============================================================
# TUKEY HSD FUNCTION
# ============================================================

def perform_tukey(
    dataframe,
    group_column,
    output_filename
):

    # =====================================================
    # CONVERT GROUP COLUMN TO STRING
    # =====================================================

    dataframe[group_column] = (
        dataframe[group_column]
        .astype(str)
    )

    # =====================================================
    # REMOVE MISSING VALUES
    # =====================================================

    dataframe = dataframe.dropna(
        subset=['MTBF_Days', group_column]
    )

    # =====================================================
    # TUKEY HSD
    # =====================================================

    tukey = pairwise_tukeyhsd(

        endog=dataframe['MTBF_Days'],
        groups=dataframe[group_column],
        alpha=0.05

    )

    # =====================================================
    # CONVERT RESULTS TO DATAFRAME
    # =====================================================

    tukey_df = pd.DataFrame(

        data=tukey.summary().data[1:],
        columns=tukey.summary().data[0]

    )

    # =====================================================
    # SAVE RESULTS
    # =====================================================

    save_path = os.path.join(
        base_output_folder,
        "Tukey_HSD",
        output_filename
    )

    tukey_df.to_excel(save_path, index=False)

    print(f"\nTukey HSD Saved : {output_filename}")

# ============================================================
# REGION-WISE TUKEY
# ============================================================

perform_tukey(
    region_monthly,
    'Region',
    'Region_Wise_Tukey.xlsx'
)

# ============================================================
# SLOC-WISE TUKEY
# ============================================================

perform_tukey(
    sloc_monthly,
    'SLoc',
    'SLoc_Wise_Tukey.xlsx'
)

# ============================================================
# CHI-SQUARE + CRAMER'S V FUNCTION
# ============================================================

def chi_square_analysis(
    dataframe,
    output_filename
):

    contingency_table = pd.crosstab(

        dataframe['Temperature_Band'],
        dataframe['MTBF_Category']

    )

    # --------------------------------------------------------
    # CHI-SQUARE
    # --------------------------------------------------------

    chi2, p, dof, expected = chi2_contingency(
        contingency_table
    )

    # --------------------------------------------------------
    # CRAMER'S V
    # --------------------------------------------------------

    n = contingency_table.sum().sum()

    cramers_v = np.sqrt(
        chi2 / (
            n * (
                min(contingency_table.shape) - 1
            )
        )
    )

    # --------------------------------------------------------
    # RESULTS DATAFRAME
    # --------------------------------------------------------

    results_df = pd.DataFrame({

        'Chi2_Statistic': [chi2],
        'P_Value': [p],
        'Degrees_of_Freedom': [dof],
        'Cramers_V': [cramers_v]

    })

    # --------------------------------------------------------
    # SAVE CHI-SQUARE
    # --------------------------------------------------------

    chi_square_path = os.path.join(
        base_output_folder,
        "Chi_Square",
        output_filename
    )

    results_df.to_excel(
        chi_square_path,
        index=False
    )

    # --------------------------------------------------------
    # SAVE CRAMER'S V
    # --------------------------------------------------------

    cramers_v_path = os.path.join(
        base_output_folder,
        "Cramers_V",
        output_filename
    )

    results_df.to_excel(
        cramers_v_path,
        index=False
    )

    print(f"\nChi-Square & Cramer's V Saved : {output_filename}")

# ============================================================
# CHI-SQUARE ANALYSIS
# ============================================================

chi_square_analysis(
    df,
    "Temperature_vs_MTBF_ChiSquare.xlsx"
)

# ============================================================
# SAVE FINAL MASTER DATASET
# ============================================================

final_output = os.path.join(
    base_output_folder,
    "Final_Temperature_Monthly_Analysis.xlsx"
)

df.to_excel(final_output, index=False)

print("\n===================================================")
print("MONTHLY TEMPERATURE ANALYSIS COMPLETED SUCCESSFULLY")
print("===================================================")

Folders Created Successfully

Dataset Loaded Successfully
Shape : (9805, 21)

Columns:
['Material', 'Equipment', 'SLoc', 'Location', 'Region', 'Season', 'Month', 'Year', 'Pstng Date', 'Next_Failure_Date', 'ON_Time_Days', 'ON_Time_Months', 'MTBF_Days', 'Tavg', 'Tmax', 'Tmin', 'RH', 'Delta_T', 'Failure_Interval_No', 'MTBF_Category', 'Temperature_Band']

All Required Columns Present

After Removing Missing Values:
(9805, 21)

Cleaned Dataset Saved

Region-Wise Monthly Dataset Saved

SLoc-Wise Monthly Dataset Saved

Correlation Results Saved : Region_Wise_Correlation.xlsx

Correlation Results Saved : SLoc_Wise_Correlation.xlsx

ANOVA Saved : Region_Wise_ANOVA.xlsx

ANOVA Saved : SLoc_Wise_ANOVA.xlsx

Kruskal Saved : Region_Wise_Kruskal.xlsx

Kruskal Saved : SLoc_Wise_Kruskal.xlsx

Tukey HSD Saved : Region_Wise_Tukey.xlsx

Tukey HSD Saved : SLoc_Wise_Tukey.xlsx

Chi-Square & Cramer's V Saved : Temperature_vs_MTBF_ChiSquare.xlsx

MONTHLY TEMPERATURE ANALYSIS COMPLETED SUCCESSFULLY


In [ ]:
# ============================================================
# SEASONAL TEMPERATURE vs MTBF ANALYSIS
# ============================================================
# DATASET:
# PCB_MTBF_Processed.xlsx
#
# ANALYSIS INCLUDED:
#
# 1. Region-wise Seasonal Correlation
# 2. SLoc-wise Seasonal Correlation
# 3. Pearson Correlation
# 4. Spearman Correlation
# 5. Kendall Tau Correlation
# 6. ANOVA
# 7. Kruskal Wallis
# 8. Tukey HSD
# 9. Chi Square Test
# 10. Cramer's V
#
# ============================================================

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

from scipy.stats import (
    pearsonr,
    spearmanr,
    kendalltau,
    f_oneway,
    kruskal,
    chi2_contingency
)

from statsmodels.stats.multicomp import pairwise_tukeyhsd

warnings.filterwarnings("ignore")

# ============================================================
# CREATE OUTPUT FOLDERS
# ============================================================

base_output_folder = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\100\MTBF\Temperature_Analysis\Seasonal_Analysis"

folders = [
    "Region_Wise",
    "SLoc_Wise",
    "Correlation_Results",
    "ANOVA",
    "Kruskal_Wallis",
    "Tukey_HSD",
    "Chi_Square",
    "Cramers_V"
]

for folder in folders:

    os.makedirs(
        os.path.join(base_output_folder, folder),
        exist_ok=True
    )

print("Folders Created Successfully")

# ============================================================
# LOAD DATASET
# ============================================================

file_path = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\100\MTBF\PCB_MTBF_Processed.xlsx"

df = pd.read_excel(file_path)

print("\nDataset Loaded Successfully")
print("Dataset Shape :", df.shape)

# ============================================================
# COLUMN STANDARDIZATION
# ============================================================

df.columns = df.columns.str.strip()

# ============================================================
# CONVERT IMPORTANT COLUMNS TO STRING
# ============================================================

df['Region'] = df['Region'].astype(str)
df['SLoc'] = df['SLoc'].astype(str)
df['Season'] = df['Season'].astype(str)

# ============================================================
# REQUIRED COLUMNS
# ============================================================

required_columns = [
    'Region',
    'SLoc',
    'Season',
    'Year',
    'MTBF_Days',
    'Tavg',
    'Temperature_Band',
    'MTBF_Category'
]

# ============================================================
# CHECK MISSING COLUMNS
# ============================================================

missing_cols = [
    col for col in required_columns
    if col not in df.columns
]

if len(missing_cols) > 0:

    print("\nMissing Columns:")
    print(missing_cols)

else:

    print("\nAll Required Columns Present")

# ============================================================
# REMOVE MISSING VALUES
# ============================================================

df = df.dropna(
    subset=[
        'MTBF_Days',
        'Tavg',
        'Region',
        'SLoc',
        'Season'
    ]
)

print("\nAfter Removing Missing Values:")
print(df.shape)

# ============================================================
# SAVE CLEANED DATASET
# ============================================================

cleaned_output = os.path.join(
    base_output_folder,
    "Temperature_Seasonal_Cleaned_Data.xlsx"
)

df.to_excel(cleaned_output, index=False)

print("\nCleaned Dataset Saved")

# ============================================================
# REGION-WISE SEASONAL DATASET
# ============================================================

region_seasonal = (
    df.groupby(
        ['Region', 'Season', 'Year']
    )
    .agg({
        'MTBF_Days': 'mean',
        'Tavg': 'mean'
    })
    .reset_index()
)

region_output = os.path.join(
    base_output_folder,
    "Region_Wise",
    "Region_Wise_Seasonal_Data.xlsx"
)

region_seasonal.to_excel(
    region_output,
    index=False
)

print("\nRegion-Wise Seasonal Dataset Saved")

# ============================================================
# SLOC-WISE SEASONAL DATASET
# ============================================================

sloc_seasonal = (
    df.groupby(
        ['SLoc', 'Season', 'Year']
    )
    .agg({
        'MTBF_Days': 'mean',
        'Tavg': 'mean'
    })
    .reset_index()
)

sloc_output = os.path.join(
    base_output_folder,
    "SLoc_Wise",
    "SLoc_Wise_Seasonal_Data.xlsx"
)

sloc_seasonal.to_excel(
    sloc_output,
    index=False
)

print("\nSLoc-Wise Seasonal Dataset Saved")

# ============================================================
# CORRELATION FUNCTION
# ============================================================

def correlation_analysis(
    dataframe,
    group_column,
    output_filename
):

    results = []

    grouped = dataframe.groupby(group_column)

    for group_name, group_data in grouped:

        if len(group_data) < 3:
            continue

        x = group_data['Tavg']
        y = group_data['MTBF_Days']

        # ====================================================
        # PEARSON
        # ====================================================

        try:
            pearson_corr, pearson_p = pearsonr(x, y)

        except:
            pearson_corr, pearson_p = np.nan, np.nan

        # ====================================================
        # SPEARMAN
        # ====================================================

        try:
            spearman_corr, spearman_p = spearmanr(x, y)

        except:
            spearman_corr, spearman_p = np.nan, np.nan

        # ====================================================
        # KENDALL TAU
        # ====================================================

        try:
            kendall_corr, kendall_p = kendalltau(x, y)

        except:
            kendall_corr, kendall_p = np.nan, np.nan

        # ====================================================
        # STORE RESULTS
        # ====================================================

        results.append({

            group_column: group_name,

            'Pearson_Correlation': pearson_corr,
            'Pearson_P_Value': pearson_p,

            'Spearman_Correlation': spearman_corr,
            'Spearman_P_Value': spearman_p,

            'KendallTau_Correlation': kendall_corr,
            'KendallTau_P_Value': kendall_p
        })

    results_df = pd.DataFrame(results)

    save_path = os.path.join(
        base_output_folder,
        "Correlation_Results",
        output_filename
    )

    results_df.to_excel(save_path, index=False)

    print(f"\nCorrelation Results Saved : {output_filename}")

# ============================================================
# REGION-WISE CORRELATION
# ============================================================

correlation_analysis(
    region_seasonal,
    'Region',
    'Region_Wise_Seasonal_Correlation.xlsx'
)

# ============================================================
# SLOC-WISE CORRELATION
# ============================================================

correlation_analysis(
    sloc_seasonal,
    'SLoc',
    'SLoc_Wise_Seasonal_Correlation.xlsx'
)

# ============================================================
# ANOVA FUNCTION
# ============================================================

def perform_anova(
    dataframe,
    group_column,
    output_filename
):

    grouped_data = []

    for name, group in dataframe.groupby(group_column):

        values = group['MTBF_Days'].dropna()

        if len(values) > 1:
            grouped_data.append(values)

    if len(grouped_data) < 2:

        print(f"Not enough groups for ANOVA : {group_column}")
        return

    f_stat, p_value = f_oneway(*grouped_data)

    anova_df = pd.DataFrame({

        'F_Statistic': [f_stat],
        'P_Value': [p_value]

    })

    save_path = os.path.join(
        base_output_folder,
        "ANOVA",
        output_filename
    )

    anova_df.to_excel(save_path, index=False)

    print(f"\nANOVA Saved : {output_filename}")

# ============================================================
# REGION-WISE ANOVA
# ============================================================

perform_anova(
    region_seasonal,
    'Region',
    'Region_Wise_Seasonal_ANOVA.xlsx'
)

# ============================================================
# SLOC-WISE ANOVA
# ============================================================

perform_anova(
    sloc_seasonal,
    'SLoc',
    'SLoc_Wise_Seasonal_ANOVA.xlsx'
)

# ============================================================
# KRUSKAL WALLIS FUNCTION
# ============================================================

def perform_kruskal(
    dataframe,
    group_column,
    output_filename
):

    grouped_data = []

    for name, group in dataframe.groupby(group_column):

        values = group['MTBF_Days'].dropna()

        if len(values) > 1:
            grouped_data.append(values)

    if len(grouped_data) < 2:

        print(f"Not enough groups for Kruskal : {group_column}")
        return

    h_stat, p_value = kruskal(*grouped_data)

    kruskal_df = pd.DataFrame({

        'H_Statistic': [h_stat],
        'P_Value': [p_value]

    })

    save_path = os.path.join(
        base_output_folder,
        "Kruskal_Wallis",
        output_filename
    )

    kruskal_df.to_excel(save_path, index=False)

    print(f"\nKruskal Saved : {output_filename}")

# ============================================================
# REGION-WISE KRUSKAL
# ============================================================

perform_kruskal(
    region_seasonal,
    'Region',
    'Region_Wise_Seasonal_Kruskal.xlsx'
)

# ============================================================
# SLOC-WISE KRUSKAL
# ============================================================

perform_kruskal(
    sloc_seasonal,
    'SLoc',
    'SLoc_Wise_Seasonal_Kruskal.xlsx'
)

# ============================================================
# TUKEY HSD FUNCTION
# ============================================================

def perform_tukey(
    dataframe,
    group_column,
    output_filename
):

    dataframe[group_column] = (
        dataframe[group_column]
        .astype(str)
    )

    dataframe = dataframe.dropna(
        subset=['MTBF_Days', group_column]
    )

    tukey = pairwise_tukeyhsd(

        endog=dataframe['MTBF_Days'],
        groups=dataframe[group_column],
        alpha=0.05

    )

    tukey_df = pd.DataFrame(

        data=tukey.summary().data[1:],
        columns=tukey.summary().data[0]

    )

    save_path = os.path.join(
        base_output_folder,
        "Tukey_HSD",
        output_filename
    )

    tukey_df.to_excel(save_path, index=False)

    print(f"\nTukey HSD Saved : {output_filename}")

# ============================================================
# REGION-WISE TUKEY
# ============================================================

perform_tukey(
    region_seasonal,
    'Region',
    'Region_Wise_Seasonal_Tukey.xlsx'
)

# ============================================================
# SLOC-WISE TUKEY
# ============================================================

perform_tukey(
    sloc_seasonal,
    'SLoc',
    'SLoc_Wise_Seasonal_Tukey.xlsx'
)

# ============================================================
# CHI-SQUARE + CRAMER'S V FUNCTION
# ============================================================

def chi_square_analysis(
    dataframe,
    output_filename
):

    contingency_table = pd.crosstab(

        dataframe['Temperature_Band'],
        dataframe['MTBF_Category']

    )

    # ========================================================
    # CHI-SQUARE
    # ========================================================

    chi2, p, dof, expected = chi2_contingency(
        contingency_table
    )

    # ========================================================
    # CRAMER'S V
    # ========================================================

    n = contingency_table.sum().sum()

    cramers_v = np.sqrt(

        chi2 / (
            n * (
                min(contingency_table.shape) - 1
            )
        )
    )

    results_df = pd.DataFrame({

        'Chi2_Statistic': [chi2],
        'P_Value': [p],
        'Degrees_of_Freedom': [dof],
        'Cramers_V': [cramers_v]

    })

    # ========================================================
    # SAVE CHI-SQUARE
    # ========================================================

    chi_path = os.path.join(
        base_output_folder,
        "Chi_Square",
        output_filename
    )

    results_df.to_excel(
        chi_path,
        index=False
    )

    cramers_path = os.path.join(
        base_output_folder,
        "Cramers_V",
        output_filename
    )

    results_df.to_excel(
        cramers_path,
        index=False
    )

    print(f"\nChi-Square & Cramer's V Saved : {output_filename}")

# ============================================================
# GLOBAL CHI-SQUARE ANALYSIS
# ============================================================

chi_square_analysis(
    df,
    "Temperature_vs_MTBF_Seasonal_ChiSquare.xlsx"
)

# ============================================================
# SAVE FINAL DATASET
# ============================================================

final_output = os.path.join(
    base_output_folder,
    "Final_Temperature_Seasonal_Analysis.xlsx"
)

df.to_excel(final_output, index=False)

# ============================================================
# COMPLETED
# ============================================================

print("\n===================================================")
print("SEASONAL TEMPERATURE ANALYSIS COMPLETED SUCCESSFULLY")
print("===================================================")

Folders Created Successfully

Dataset Loaded Successfully
Dataset Shape : (9805, 21)

All Required Columns Present

After Removing Missing Values:
(9805, 21)

Cleaned Dataset Saved

Region-Wise Seasonal Dataset Saved

SLoc-Wise Seasonal Dataset Saved

Correlation Results Saved : Region_Wise_Seasonal_Correlation.xlsx

Correlation Results Saved : SLoc_Wise_Seasonal_Correlation.xlsx

ANOVA Saved : Region_Wise_Seasonal_ANOVA.xlsx

ANOVA Saved : SLoc_Wise_Seasonal_ANOVA.xlsx

Kruskal Saved : Region_Wise_Seasonal_Kruskal.xlsx

Kruskal Saved : SLoc_Wise_Seasonal_Kruskal.xlsx

Tukey HSD Saved : Region_Wise_Seasonal_Tukey.xlsx

Tukey HSD Saved : SLoc_Wise_Seasonal_Tukey.xlsx

Chi-Square & Cramer's V Saved : Temperature_vs_MTBF_Seasonal_ChiSquare.xlsx

SEASONAL TEMPERATURE ANALYSIS COMPLETED SUCCESSFULLY
